In [0]:
from pyspark.sql.functions import *

# ==========================================================
# Read Tables
# ==========================================================

pde = spark.read.table(
    "healthcare_claims_catalog.silver.pde"
)

dim_beneficiary = spark.read.table(
    "healthcare_claims_catalog.gold.dim_beneficiary"
)

dim_date = spark.read.table(
    "healthcare_claims_catalog.gold.dim_date"
)

# ==========================================================
# Join Dimensions
# ==========================================================

fact_pharmacy = (

    pde

    .join(
        dim_beneficiary.select(
            "DESYNPUF_ID",
            "BENEFICIARY_KEY"
        ),
        "DESYNPUF_ID",
        "left"
    )

)

# ==========================================================
# Date Key
# ==========================================================

fact_pharmacy = fact_pharmacy.withColumn(

    "DATE_KEY",

    date_format(
        "SRVC_DT",
        "yyyyMMdd"
    ).cast("int")

)

# ==========================================================
# Insurance Payment
# ==========================================================

fact_pharmacy = fact_pharmacy.withColumn(

    "INSURANCE_PAYMENT",

    greatest(

        lit(0),

        col("TOT_RX_CST_AMT")
        -
        col("PTNT_PAY_AMT")

    )

)

# ==========================================================
# Average Cost Per Day
# ==========================================================

fact_pharmacy = fact_pharmacy.withColumn(

    "COST_PER_DAY",

    when(

        col("DAYS_SUPLY_NUM")>0,

        round(
            col("TOT_RX_CST_AMT")/
            col("DAYS_SUPLY_NUM"),
            2
        )

    )

)

# ==========================================================
# Select Columns
# ==========================================================

fact_pharmacy = fact_pharmacy.select(

    col("PDE_ID"),

    col("BENEFICIARY_KEY"),

    col("DATE_KEY"),

    col("PROD_SRVC_ID"),

    col("QTY_DSPNSD_NUM"),

    col("DAYS_SUPLY_NUM"),

    col("TOT_RX_CST_AMT").alias("TOTAL_DRUG_COST"),

    col("PTNT_PAY_AMT").alias("PATIENT_PAYMENT"),

    col("INSURANCE_PAYMENT"),

    col("COST_PER_DAY"),

    current_timestamp().alias("GOLD_CREATED_TIMESTAMP")

)

# ==========================================================
# Write
# ==========================================================

fact_pharmacy.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.format("delta") \
.saveAsTable(
"healthcare_claims_catalog.gold.fact_pharmacy"
)

# ==========================================================
# Validation
# ==========================================================

print("="*60)
print("FACT PHARMACY CREATED")
print("="*60)

print("Records :",fact_pharmacy.count())

fact_pharmacy.show(10,False)

FACT PHARMACY CREATED
Records : 16540128
+---------------+---------------+--------+------------+--------------+--------------+---------------+---------------+-----------------+------------+--------------------------+
|PDE_ID         |BENEFICIARY_KEY|DATE_KEY|PROD_SRVC_ID|QTY_DSPNSD_NUM|DAYS_SUPLY_NUM|TOTAL_DRUG_COST|PATIENT_PAYMENT|INSURANCE_PAYMENT|COST_PER_DAY|GOLD_CREATED_TIMESTAMP    |
+---------------+---------------+--------+------------+--------------+--------------+---------------+---------------+-----------------+------------+--------------------------+
|376014477111060|221113         |20081105|67253022510 |270.00        |10            |10.00          |0.00           |10.00            |1.00        |2026-08-06 18:38:26.863895|
|376014477111812|171482         |20090901|57866493901 |90.00         |10            |240.00         |0.00           |240.00           |24.00       |2026-08-06 18:38:26.863895|
|376014477112047|144198         |20080907|00173073601 |60.00         |30       